<a href="https://colab.research.google.com/github/akinlayoadeniran/marketing-analytics-ml/blob/main/Akin_Generative_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

def generate_text(prompt, max_new_tokens=30, temperature=0.8):
    inputs = tokenizer(prompt, return_tensors='pt')
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

#Exercise 1

In [16]:
prompt = 'Positive customer review: '
print(generate_text(prompt))

Positive customer review:  I have only had this product for a couple of months and had nothing but positive reviews. Even the reviewers who have not been doing reviews have given


If it is not a good fit, go for the best option. a little bit more expensive. Rated 5.        Mixed, and it mentions a price complaint.

The packaging is pretty cute. there was a picture of a guy in his 40.     Off-topic, drift into saying something elese.

I just picked up my guitar. It is not great and the front is not solid.     No, this is negative.

I have only had this product for a couple of months and had nothing but positive review.       Closest to a positive review.




The model often ignores the word “Positive”. Run 3 is actually negative.

It sometimes drifts away from the topic (the guitar and the “guy in his 40” were never mentioned in the prompt).

Every output is cut off mid-sentence (“.in his 40”, “.All in all it is a”), because max_new_tokens=30 stops it after 30 tokens.

#Exercise 2

In [19]:
prompt = 'Write a detailed positive review for a laptop'
print(generate_text(prompt))

Write a detailed positive review for a laptop computer.

How to check a computer's quality:

If you have a PC that is not good enough or that is too big for


For a laptop you own. If you are looking for a great laptop and you know that it is going to cost you money, you are probably not trying.
Not really, it talking about cost.

For a laptop. The PCWorld review is a fun way to get a sense of just how different the hardware is.   
No, it is talking about a PCWorld review instead of reviewing a

for a laptop computer. How to check a computer quality: If you have a PC that is not good enough.
No, it turns into a how to guide on PC.

In [21]:
prompt = 'This laptop is amazing. The battery life is excellent and'
print(generate_text(prompt))

This laptop is amazing. The battery life is excellent and the computer is very easy to manage. I have been using this laptop for around 6 months now and have never had any trouble with the laptop. The


The portability is solid. I’m not sure what to expect after getting this laptop, but it’s... Rated 4
Yes, mostly positive and on topic

The computer is very easy to manage. I have been using this laptop for around 6 months now.
Yes, on topic, positive, and sounds like a real review

The instruction prompt (“Write a detailed positive review for a laptop”) made GPT-2 repeat the instruction and drift into other topics. A prompt that starts the review (“This laptop is amazing. The battery life is excellent and”) produced on-topic positive reviews, because GPT-2 only continues text and doesn’t follow commands.

#Exercise 3

In [24]:
print('Low temp:')
print(generate_text('Product review:', temperature=0.3))
print('\nHigh temp:')
print(generate_text('Product review:', temperature=1.2))

Low temp:
Product review:

The following review is for the new version of the game.

The following review is for the new version of the game.



High temp:
Product review:

My experience is that when using the 2-part kit, its important that you use a minimum of two products which should be used separately so


A made-up TV show description, off-topic.
No reviews present (Visited 2,996 times.), text copied from a web page.

The first time I tried this I was so excited. So happy with the, a coherent review.
My question goes without saying, grammatical but meaningless.

The following review is for the new version of the game. repeated twice
My experience is that when using the 2-part kit, plausible but random.


Low temperature (0.3) made GPT-2 more conservative and sometimes repetitive (“The following review is for the new version of the game” twice). High temperature (1.2) made it more random and less coherent, with text that was off-topic or meaningless. Even at low temperature the outputs differed between runs, because sampling is still random. Temperature controls how risky the word choices are, not whether they are random.

#Exercise 4

In [26]:
prompt = 'Rewrite politely: bad product and slow delivery'
print(generate_text(prompt))

Rewrite politely: bad product and slow delivery

Replace the "Firmware Update" prompt with:

Firmware Update

Install Windows Server 2012 R2




Rewrite politely: bad product and slow delivery. You want to do well here, but not here at all. You want to do well at a place where customers are a.
No, it repeated the prompt and rambled about customers.

Rewrite politely: bad product and slow delivery. Replace the ‘Firmware Update’ prompt with: Firmware Update, Install Windows Server 2012 R2.
No, it drifted into unrelated computer text.


GPT-2 did not rewrite the sentence politely. It repeated the instruction and then continued with unrelated text (customer service rambling, then a firmware update). A rewriting task is an instruction, and GPT-2 only predicts the next words, so it can’t carry out the task.

In [28]:
prompt = 'Rude review: bad product and slow delivery.\nPolite version:'
print(generate_text(prompt))

Rude review: bad product and slow delivery.
Polite version: The new model is a little too stiff for me. I got one of these two days ago and was pleasantly surprised by how good it is. I


excellent quality. Pros/Cons: (1) No reviews. (2) Not in stock. (3) I purchased this.
No. It flipped the meaning to “excellent quality” and then drifted into a pros/cons list.

“The new model is a little too stiff for me. I got one of these two days ago and was pleasantly surprised by how good it is. I”.
No. It’s polite in tone, but it’s about a different product and contradicts itself (“too stiff” then “pleasantly surprised”)

Giving GPT-2 a pattern (“Rude review: ... Polite version:”) worked slightly better than a direct instruction. The outputs sounded like polite reviews, but they were not rewrites of the original sentence. One changed the meaning to “excellent quality” and the other described a different product and contradicted itself. GPT-2 imitates the style of text but doesn’t understand or preserve meaning.

In [31]:
prompt = 'Once upon a time, in a small village in Portugal,'
print(generate_text(prompt))

Once upon a time, in a small village in Portugal, a girl was playing in the woods when a big dog suddenly went up for cover beside her. The girl screamed and ran to safety, but before she


The people of the region called the local population ‘Monsieur de Mors.’ They called their people.
Stays in the story but invents a French-sounding name for a Portuguese village, which is a factual mistake.

A girl was playing in the woods when a big dog suddenly went up for cover beside her. The girl screamed and ran to safety, but before she.
Reads like a story opening, grammatical and on topic, but the “dog went up for cover” is odd.

I tried the prompt “Once upon a time, in a small village in Portugal,”. GPT-2 handled this better than the reviews and the rewriting task, because the prompt starts a story and the model only has to continue it. Both outputs were fluent and on topic, but they contained invented or odd details (a French-sounding name for a Portuguese village, a dog “going up for cover”). GPT-2 produces plausible-sounding text but doesn’t check whether it is true or makes sense.

1. Which prompt worked best?
The one that worked best was “This laptop is amazing. The battery life is excellent and” from Exercise 2. Both times I ran it, I got a positive review about the laptop, for example “the computer is very easy to manage” and “I have been using this laptop for around 6 months now”. The instruction “Write a detailed positive review for a laptop” did not work. The model just repeated it and then started talking about a PCWorld review and how to check a computer’s quality. GPT-2 only continues text, it does not follow commands, so it did better when I started the review myself. The story opening in Exercise 5 worked for the same reason.

2. How does temperature affect output?
Temperature changes how many risks the model takes with the next word. At 0.3 it played it safe. Sometimes I got a normal review, but in one run it repeated “The following review is for the new version of the game” twice. At 1.2 the text was more random. One output looked like it was copied from a web page (“No reviews present, visited 2,996 times”), and another was a proper sentence that did not mean anything. Even at 0.3 I got something different every time, because the model still picks words partly by chance. Temperature only changes how adventurous the choices are.

3. What are the limitations of the model?
GPT-2 can not follow instructions, so the “rewrite politely” exercise failed. It often goes off topic. My reviews turned into guitars, firmware updates and Windows Server. It also makes things up, like calling a Portuguese village “Monsieur de Mors” and having a dog “go up for cover”. At low temperature it repeats itself. It does not understand meaning either. When I asked for a polite version of a bad review, one output said “excellent quality”, which is the opposite of the complaint. The outputs also get cut off mid-sentence because of the 30 token limit, and the results change every time I run it, so you can not rely on them.